# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR² Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a [Croissant schema](https://mlcommons.org/croissant/) available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

This notebook provides a guided example for FAIR^2 datasets, referencing all entities by their Croissant `@id`.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the metadata and instantiate the `mlcroissant` dataset from the Croissant schema.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Show the top-level metadata fields
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Keywords: {', '.join(dataset.metadata.keywords)}")


## 2. Data Overview
Review dataset record sets and fields. All entities are referenced using their `@id` attribute for transparency and reproducibility.


In [ ]:
# List available record sets in the dataset
print('Available Record Sets:')
record_sets = dataset.metadata.recordSets
for record_set in record_sets:
    print(f"- @id: {record_set.id}")
    print(f"  name: {getattr(record_set, 'name', '')}")
    print(f"  description: {getattr(record_set, 'description', '')}\n")

# For the sake of the example, pick the first record set for further exploration
record_set_id = record_sets[0].id
print(f"\nFirst RecordSet chosen for demonstration: {record_set_id}")

# List all fields and their @id in the chosen record set
fields = record_sets[0].fields
print('Fields in the chosen RecordSet:')
for field in fields:
    print(f"- @id: {field.id}")
    print(f"  name: {getattr(field, 'name', '')}")
    print(f"  data type: {getattr(field, 'dataType', '')}")
    print(f"  description: {getattr(field, 'description', '')}\n")

# Optionally, sample the first few records
print(f"Sample records from RecordSet {record_set_id}:")
for i, record in enumerate(dataset.records(record_set=record_set_id)):
    print(record)
    if i >= 2:
        break

## 3. Data Extraction
Load the records from record sets into pandas DataFrames for further analysis. You can extend this list by referencing additional record set `@id`s.


In [ ]:
# Use all record set @id's for extraction
record_set_ids = [rs.id for rs in dataset.metadata.recordSets]
dataframes = {}

# Load each record set into a pandas DataFrame
for rs_id in record_set_ids:
    # Ensure to load only tabular record sets
    df = pd.DataFrame(list(dataset.records(record_set=rs_id)))
    dataframes[rs_id] = df
    print(f"Loaded DataFrame for RecordSet @id: {rs_id}. Shape: {df.shape}")
    print(df.columns.tolist())
    print(df.head(2), '\n')

# For the rest of the steps, we'll use the first RecordSet
df_main = dataframes[record_set_id]

## 4. Exploratory Data Analysis (EDA)
Select a numeric field by its `@id` and demonstrate typical numeric cleaning/EDA steps, such as filtering, normalization, and grouping. All field references use their Croissant `@id`.


In [ ]:
# Get all numeric fields in the chosen RecordSet
import numbers

numeric_field_id = None
fields = [f for f in dataset.metadata.recordSets[0].fields]
for f in fields:
    # Croissant dataType is typically a URL or a short name such as 'Float', 'Integer', etc.
    if getattr(f, 'dataType', None) in ['Float', 'Integer', 'Number']:
        numeric_field_id = f.id
        print(f"Numeric field selected for EDA: {numeric_field_id} ({getattr(f, 'name', '')})")
        break

if numeric_field_id is None:
    print("No numeric fields found.")
else:
    # If columns are not named by @id, try to match
    column = numeric_field_id
    if column not in df_main.columns:
        # Some datasets export with only short column names; try stripping to the field name
        column = getattr([f for f in fields if f.id == numeric_field_id][0], 'name')

    # Drop missing/inconvertible values
    df_eda = df_main.copy()
    df_eda[column] = pd.to_numeric(df_eda[column], errors='coerce')
    df_eda = df_eda.dropna(subset=[column])

    threshold = df_eda[column].mean()  # Example: threshold at mean
    filtered_df = df_eda[df_eda[column] > threshold]

    print(f"Filtered records with {column} > {threshold:.2f} (Field @id: {numeric_field_id}):")
    print(filtered_df.head())

    filtered_df[f"{column}_normalized"] = (
        (filtered_df[column] - filtered_df[column].mean()) /
        filtered_df[column].std()
    )
    print(f"\nNormalized {column} (z-score):")
    print(filtered_df[[column, f"{column}_normalized"]].head())

    # Try to pick a groupable (categorical) field for grouping, such as one with string data
    group_field_id = None
    for f in fields:
        if getattr(f, 'dataType', None) in ['Text', 'String']:
            group_field_id = f.id
            if group_field_id in filtered_df.columns:
                break

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[column].mean().reset_index()
        print(f"\nMean {column} grouped by {group_field_id}:")
        print(grouped_df.head())
    else:
        print('No categorical grouping field found for this analysis.')

## 5. Visualization
Explore the numeric field distribution and a possible relationship with a categorical/group field, using `matplotlib` or `seaborn` as appropriate.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and column in df_eda.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df_eda[column], kde=True)
    plt.title(f'Distribution of {column} (Field @id: {numeric_field_id})')
    plt.xlabel(column)
    plt.ylabel("Count")
    plt.show()
    
    if group_field_id and group_field_id in df_eda.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=df_eda, x=group_field_id, y=column)
        plt.title(f'{column} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(column)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

- This notebook demonstrates loading and exploring a Croissant FAIR² dataset with `mlcroissant`.
- All manipulations are referenced by Croissant `@id`.
- The approach can be adapted for any FAIR² or Croissant dataset for standard data science workflows.
